<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>

# Part 1 Assessment

You get a new 30-second traffic clip and work with it on your own. Seven short tasks use VIOS (video
storage), RT-CV (detection and tracking), Kafka, Elasticsearch, RT-VLM (captioning) and RT-Embed (video
search). Eight multiple-choice questions follow.

In this notebook you will:

- add a video to VIOS and read RT-CV's detections from Elasticsearch and from Kafka
- caption the clip with RT-VLM at two chunk sizes and with more sampled frames
- find a collision with RT-Embed, and find a car by its appearance with RT-CV
- work out what captioning one camera for a day costs RT-VLM

**Time:** 25 minutes.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Part 1** &middot; about 25 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> **Before you start:** notebooks 1 to 4 done; the lab pill is green.
>
> Previous: [4. RT-Embed: Video Search](04-rt-embed-video-search.ipynb) &middot; Next: VSS Coding Agent Skills (open Course Docs from the JupyterLab Launcher)


## How this assessment works

Work on your own. Keep notebooks 1 to 4 open: every call you need is written out there.

Each task has an instruction, a code cell for you to complete and a check cell. Replace every `FIXME` in the
code cell, run it, then run the check. The check prints **PASS** or **NOT YET** with the reason. Fix your code
and run both cells again as often as you like. This notebook has no solution cells.

You pass Part 1 when:

- tasks T1, T2, T4 and T5 pass, and
- at least 6 of the 8 questions are answered correctly.

T3, T6 and T7 are reported but do not decide the result. The T5 check also says whether your window shows a labelled
collision, and that line does not decide the result either. Each check cell saves your answer in
`/dli/task/work/assessment/answers.json`. The Submit cell at the end asks the grader to read that file and
check your work again against the live services.

## Set up and cut your clip

The next cell loads the course helpers and cuts 30 seconds out of `rain_eval.mp4`, one of the four traffic
videos from notebook 4. It saves the cut as `/dli/task/work/assessment/rain_check.mp4`. The deployment has not
processed this file yet. RT-CV will label its records with the file stem, `rain_check`. If VIOS already holds
the clip from an earlier run, the cell says so, and you skip the upload in T1.

Watch the clip once before you start.

In [ ]:
import json, subprocess, sys
from pathlib import Path

import requests

import course_setup as cs      # the shared VSS helpers from notebooks 2 to 4
import labtools                # video and drawing helpers
import quiz                    # the checks and the questions of this assessment
from cleanup import remember   # the Part 1 cleanup ledger

BASE = cs.BASE                                       # http://nginx: the lab reaches every service through it
SOURCE_VIDEO = Path("/dli/task/data/search-videos/rain_eval.mp4")
WORK = quiz.WORK                                     # /dli/task/work/assessment
CLIP = WORK / "rain_check.mp4"
STEM = CLIP.stem                                     # rain_check: the name RT-CV gives the clip
CLIP_S = 30                                          # the clip's length in seconds
ANCHOR = "2025-01-01T00:00:00.000Z"                  # where the clip starts on its VIOS timeline
UPLOAD_HEADERS = {"nvstreamer-chunk-number": "1", "nvstreamer-total-chunks": "1", "nvstreamer-is-last-chunk": "true",
                  "nvstreamer-identifier": CLIP.name, "nvstreamer-file-name": CLIP.name}

if not SOURCE_VIDEO.is_file():
    raise FileNotFoundError(f"{SOURCE_VIDEO} is missing: the course data did not load. Call the instructor.")
WORK.mkdir(parents=True, exist_ok=True)
if not CLIP.is_file():
    # Re-encode instead of copying, so the cut starts exactly at 15 s. One key frame per second keeps VIOS cuts precise.
    partial = CLIP.with_name("rain_check.partial.mp4")
    cut = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", "15", "-i", str(SOURCE_VIDEO), "-t", str(CLIP_S),
                          "-c:v", "libx264", "-pix_fmt", "yuv420p", "-g", "24", "-an", "-movflags", "+faststart",
                          str(partial)], capture_output=True, text=True, timeout=300)
    if cut.returncode:
        raise RuntimeError("ffmpeg could not cut the clip: " + cut.stderr.strip()[-300:])
    partial.replace(CLIP)
quiz.save("clip_stem", STEM)

found = cs.find_sensor(STEM)                         # None until T1 adds the clip to VIOS
SENSOR_ID = found["sensorId"] if found else None
print(f"clip: {CLIP} ({CLIP.stat().st_size / 1e6:.1f} MB)")
print("VIOS:", f"already lists {found['name']}, so skip the upload cell in T1" if found else "no copy yet; T1 adds it")
if not Path("/dli/task/data/search_sources.json").is_file():
    print("Note: /dli/task/data/search_sources.json is missing, and T5 needs it. Tell the instructor.")
labtools.show_video(CLIP.read_bytes())

## T1. Add the clip to VIOS

Upload `rain_check.mp4` with the three calls from notebook 2:

1. Ask the VSS agent service for an upload URL.
2. Send the file to VIOS. VIOS answers with the new source's `sensorId`.
3. Tell the VSS agent service that the upload is complete. This call starts RT-CV and RT-Embed on the file.

**Your turn:** replace the three `FIXME`s, then run the cell once. Each run adds another copy of the clip to
VIOS. If the setup cell said that VIOS already lists the clip, skip this cell and run the check. The cell
prints the new `sensorId`.

In [ ]:
# TODO
# 1. Ask the VSS agent service for an upload URL, then point it at VIOS's port inside the lab.
r = requests.post(BASE + FIXME, json={"filename": CLIP.name}, timeout=60)
r.raise_for_status()
target = cs.upload_url(r.json()["url"])

# 2. Send the file to VIOS in one chunk. The timestamp is where the clip starts on its VIOS timeline.
with CLIP.open("rb") as fh:
    r = requests.post(target, headers=UPLOAD_HEADERS, timeout=300,
                      files={"metadata": (None, json.dumps({"timestamp": ANCHOR})),
                             "file": (CLIP.name, fh, "video/mp4")})
r.raise_for_status()
SENSOR_ID = r.json()[FIXME]                    # the VIOS id of the new source
remember("video", SENSOR_ID, name=STEM)        # so the course cleanup can remove it later

# 3. Tell the VSS agent service that the upload is complete.
r = requests.post(BASE + FIXME, json={"filename": CLIP.name}, timeout=600)
r.raise_for_status()
print("VIOS sensorId:", SENSOR_ID)

In [ ]:
# Check T1: VIOS lists the clip as rain_check or rain_check_<n>, with a timeline.
vios_names = {s.get("sensorId"): s.get("name", "") for s in cs.sensors()}
quiz.save("t1", {"vios_sensor_id": SENSOR_ID, "vios_name": vios_names.get(SENSOR_ID, "")})
passed = quiz.task("p1-t1", "Add the clip to VIOS", quiz.check_t1)

## T2. Count RT-CV's frame records in Elasticsearch

RT-CV publishes one record per processed frame, and Logstash indexes it in `mdx-raw-2025-01-01`. A record
keeps the frame's detected objects as nested documents, and each object carries its class in `type`
(notebook 2).

The next cell waits until RT-CV has finished the clip. It prints the number of records every 10 seconds and
returns when the number stops growing.

In [ ]:
# Wait until RT-CV has processed the whole clip: the number of frame records must stop growing.
steady = quiz.wait_until_steady(STEM)

**Your turn:** count the frame records for `rain_check`, then find the most frequent object type among their
objects. Replace the three `FIXME`s. The cell prints the frame count, then one line per object type, most
frequent first.

In [ ]:
# TODO
clip_filter = {"term": {FIXME: STEM}}             # the field RT-CV writes the file stem to
frames = cs.es("POST", "/mdx-raw-2025-01-01/_count", json={"query": clip_filter})["count"]

result = cs.es("POST", "/mdx-raw-2025-01-01/_search", json={
    "size": 0,
    "query": clip_filter,
    "aggs": {"objects": {"nested": {"path": FIXME},                         # where a record keeps its objects
                         "aggs": {"types": {"terms": {"field": FIXME}}}}},   # the keyword field of an object's class
})
buckets = result["aggregations"]["objects"]["types"]["buckets"]
top_type = buckets[0]["key"]
print(f"{frames} frame records for {STEM}")
for b in buckets:
    print(f"  {b['key']:10} {b['doc_count']} detections")

In [ ]:
# Check T2: your count and your top type against Elasticsearch now.
quiz.save("t2", {"frames": frames, "top_type": top_type})
passed = quiz.task("p1-t2", "Count RT-CV's frame records and the top object type", quiz.check_t2)

## T3. Read a detection message from Kafka

RT-CV publishes every frame to the Kafka topic `mdx-raw` as a Protobuf `Frame` message, and Logstash indexes
the same messages in Elasticsearch (notebook 2, section 2.6). The topic holds messages from every video, split
into partitions. RT-CV sends all frames of one video to the same partition. If RT-CV has processed no other video since
yours, the newest messages of that partition are frames of your clip. The next cell reads the last 20 messages of each partition. The messages are
binary, so the cell splits them on a marker, not on line breaks.

In [ ]:
# Read the newest messages of mdx-raw: the last TAIL messages of every partition.
TAIL = 20
MARK = "<<end-of-message>>"     # printed after every message, because Protobuf bytes can contain line breaks

offsets = subprocess.run(["docker", "exec", "kafka", "kafka-get-offsets", "--bootstrap-server", "localhost:9092"],
                         capture_output=True, text=True, timeout=120)
if offsets.returncode:
    raise RuntimeError("kafka-get-offsets failed: " + offsets.stderr.strip()[-300:])
ends = {}                       # partition -> the offset after its newest message
for line in offsets.stdout.splitlines():
    parts = line.rsplit(":", 2)
    if len(parts) == 3 and parts[0] == "mdx-raw":
        ends[int(parts[1])] = int(parts[2])

payloads = []
for partition, end in sorted(ends.items()):
    count = min(TAIL, end)
    if count == 0:
        continue
    read = subprocess.run(
        ["docker", "exec", "kafka", "timeout", "30", "kafka-console-consumer", "--bootstrap-server", "localhost:9092",
         "--topic", "mdx-raw", "--partition", str(partition), "--offset", str(end - count),
         "--max-messages", str(count), "--formatter-property", "line.separator=" + MARK],
        capture_output=True, timeout=120)
    payloads += [p for p in read.stdout.split(MARK.encode()) if p]
if not payloads:
    raise RuntimeError("No messages read from mdx-raw. Check that T2 found records, then run this cell again.")
print(f"{len(payloads)} messages read from {len(ends)} partitions of mdx-raw")

**Your turn:** decode the messages with the `Frame` class from the VSS checkout, and keep the first one that
comes from `rain_check`. Replace the two `FIXME`s. The cell prints the frame id, its time and how many objects
it holds.

In [ ]:
# TODO
sys.path.insert(0, "/opt/vss/tools/message-broker-consumers")   # the Protobuf module that ships with VSS
from schema_pb2 import Frame

frame = None
for payload in payloads:
    message = Frame()
    message.FIXME(payload)                      # decode the bytes into the Frame
    if message.FIXME == STEM:                   # the field that names the source
        frame = message
        break
if frame is None:
    raise RuntimeError(f"None of the newest messages comes from {STEM}. Set TAIL = 100 in the cell above and run "
                       "both cells again. If you started the live camera in notebook 2, release it first "
                       "(notebook 2, section 2.13).")
print(f"frame {frame.id} at {frame.timestamp.ToJsonString()} with {len(frame.objects)} objects")

In [ ]:
# Check T3: the message comes from your clip, and Elasticsearch holds the same frame.
if frame is None:
    raise RuntimeError(f"Run the cell above until it finds a message from {STEM}.")
quiz.save("t3", {"sensor_id": frame.sensorId, "frame_id": str(frame.id), "timestamp": frame.timestamp.ToJsonString()})
passed = quiz.task("p1-t3", "Read and decode a detection message from Kafka", quiz.check_t3)

## T4. Caption the clip with RT-VLM

Register the whole clip with RT-VLM, then caption it three times with the same prompt:

1. One answer for every 10 seconds of video.
2. One answer for the whole clip.
3. One answer for every 10 seconds, with 8 sampled frames in every chunk. This one is a REST request that you
   complete yourself. The two frames fields from notebook 3, section 3.4, are filled in. You name the field that
   sets the chunk length, as in sections 3.2 to 3.4.

Every chunk in a response reports `frame_count`, the number of frames the model saw, and `vlm_latency_ms`, the
time the model took. Compare `frame_count` in the first and the third call.

**Your turn:** write a prompt for `PROMPT`, then replace the other three `FIXME`s. A correct run prints 3
chunks, then 1 chunk, then 3 chunks that each report 8 frames. A fourth, very short chunk at the end of the
10-second runs is fine.

In [ ]:
# TODO
PROMPT = FIXME          # a string: one specific question about the traffic in the clip

# Register the whole 30-second clip: VIOS cuts the window and RT-VLM fetches it.
file_id = cs.register_vios_window(SENSOR_ID, ANCHOR, cs.iso_offset(ANCHOR, CLIP_S), ANCHOR)
remember("caption", file_id)                    # so the course cleanup can remove it later

captions_10 = cs.caption(file_id, PROMPT, chunk_duration=FIXME)    # one answer every 10 seconds
captions_0 = cs.caption(file_id, PROMPT, chunk_duration=FIXME)     # one answer for the whole clip

# The same 10-second chunks with 8 sampled frames each, as a REST request. The two frames fields are filled in.
r = requests.post(BASE + "/rtvi-vlm/v1/generate_captions", timeout=900, json={
    "id": file_id, "model": cs.vlm_model_id(), "prompt": PROMPT,
    FIXME: 10,                                  # the field that sets the chunk length in seconds
    "num_frames_per_second_or_fixed_frames_chunk": 8,   # the frames sampled per chunk
    "use_fps_for_chunking": False,              # 8 is a number of frames per chunk, not frames per second
})
r.raise_for_status()
captions_8 = r.json()["chunk_responses"]

for label, chunks in (("chunk_duration=10", captions_10), ("chunk_duration=0", captions_0),
                      ("10 s with 8 frames", captions_8)):
    print(f"\n{label}: {len(chunks)} chunk(s)")
    for c in chunks:
        print(f"  {c.get('frame_count')} frames | {c.get('vlm_latency_ms')} ms | {c['content'].strip()[:150]}")

In [ ]:
# Check T4: 3 chunks at 10 s (4 if the cut runs a little past 30 s), 1 chunk at 0, 8 frames in each full chunk
# of the third call, and the captions stored.
quiz.save("t4", {"file_id": file_id, "chunks_10": len(captions_10), "chunks_0": len(captions_0),
                 "frame_counts_8": [c.get("frame_count") for c in captions_8]})
(WORK / "t4_captions.json").write_text(json.dumps(
    {"chunk_duration_10": captions_10, "chunk_duration_0": captions_0, "frames_8": captions_8}, indent=1) + "\n")
passed = quiz.task("p1-t4", "Caption the clip at two chunk sizes and with 8 sampled frames", quiz.check_t4)

## T5. Find a collision with RT-Embed

RT-Embed has indexed the four traffic videos from notebook 4: `normal_eval.mp4`, `rain_eval.mp4`,
`fog_eval.mp4` and `night_eval.mp4`. Search them with a text query and watch the top window.

Cosmos-Embed1, the model behind RT-Embed, learned from pairs of video and caption. Write the query the way a caption
describes the moment, in a few words (notebook 4, section 4.7). The cell turns the top hit's times into seconds from
the start of its video and plays that window from VIOS.

**Your turn:** write a query, then watch the top window. The check runs the same
search with your query and compares its top window with yours. It also says whether your window overlaps a
labelled collision. That line does not decide T5, so try a few wordings and watch what each one finds.

In [ ]:
# TODO
QUERY = FIXME            # a string: a collision in a few words, the way a caption would describe it

# Written at launch by course-prep: filename -> {"sensor_id", "creation_time", "duration_seconds", ...}
registry = json.loads(Path("/dli/task/data/search_sources.json").read_text())
files = ["normal_eval.mp4", "rain_eval.mp4", "fog_eval.mp4", "night_eval.mp4"]
by_sensor = {registry[name]["sensor_id"]: name for name in files}

hits = cs.knn_search(cs.text_embedding(QUERY), list(by_sensor), k=5)
top = hits[0]["_source"]
TOP_FILE = by_sensor[top["sensor"]["id"]]

# Seconds from the start of the video: the index starts every video at cs.EMBEDDING_ANCHOR (notebook 4, 4.6).
TOP_START_S = cs.offset_seconds(cs.EMBEDDING_ANCHOR, top["timestamp"])
TOP_END_S = cs.offset_seconds(cs.EMBEDDING_ANCHOR, top["end"])
print(f"top hit: {TOP_FILE} from {TOP_START_S:.1f} to {TOP_END_S:.1f} s, score {hits[0]['_score']:.3f}")

video_start = registry[TOP_FILE]["creation_time"]   # where VIOS starts this video
labtools.show_video(cs.clip_bytes(top["sensor"]["id"], cs.iso_offset(video_start, TOP_START_S),
                                  cs.iso_offset(video_start, TOP_END_S)))

In [ ]:
# Check T5: the same search, run again with your query, gives your window.
# Overlap with a labelled collision is reported.
quiz.save("t5", {"query": QUERY, "file": TOP_FILE, "start_s": round(TOP_START_S, 2), "end_s": round(TOP_END_S, 2)})
passed = quiz.task("p1-t5", "Find a collision with a text query", quiz.check_t5)

## T6. Find a car by its appearance

RT-CV stores an embedding with each object it detects: 1,152 numbers that describe how an object looks.
Objects that look alike have embeddings that are close, so a kNN search on the objects' embeddings finds them
(notebook 2, section 2.12). The VSS user interface calls this search by image: you pick an object in a frame, and VSS finds
objects that look like it. RT-CV has no route that embeds a photo you upload.

The next cell picks one car from your clip. It prints the car's object id and the time of its frame, draws
its box, and keeps its embedding in `QUERY_VECTOR`.

In [ ]:
# The car to search for: the largest car with an embedding in the first frame 10 seconds or more into the clip.
records = cs.es("POST", "/mdx-raw-2025-01-01/_search", json={
    "size": 24, "sort": [{"timestamp": {"order": "asc"}}],         # about one second of frames
    "query": {"bool": {"filter": [cs.source_filter(STEM),
                                  {"range": {"timestamp": {"gte": cs.iso_offset(ANCHOR, 10)}}}]}},
})["hits"]["hits"]


def box_area(obj):
    box = obj.get("bbox") or {}
    return max(0, box.get("rightX", 0) - box.get("leftX", 0)) * max(0, box.get("bottomY", 0) - box.get("topY", 0))


query_frame, query_object = None, None
for hit in records:
    cars = [o for o in hit["_source"].get("objects") or []
            if o.get("type") == "car" and (o.get("embedding") or {}).get("vector")]
    if cars:
        query_frame, query_object = hit["_source"], max(cars, key=box_area)
        break
if query_object is None:
    raise RuntimeError(f"No car with an embedding in the first records after 10 s of {STEM}. Finish T2 first.")

QUERY_OBJECT_ID = str(query_object["id"])
QUERY_FRAME_TIME = query_frame["timestamp"]
QUERY_VECTOR = query_object["embedding"]["vector"]
print(f"object {QUERY_OBJECT_ID}, a {query_object['type']}, in the frame at {QUERY_FRAME_TIME}")
print(f"its embedding has {len(QUERY_VECTOR)} values")

seconds = cs.offset_seconds(ANCHOR, QUERY_FRAME_TIME)
picture = labtools.frame_from_mp4(cs.clip_bytes(SENSOR_ID, cs.iso_offset(ANCHOR, seconds),
                                                cs.iso_offset(ANCHOR, seconds + 1)))
labtools.draw_boxes(picture, [query_object])

**Your turn:** complete the nested kNN search. It finds the object that looks most like this car in any other
frame of the clip. Replace the two `FIXME`s. The cell prints the closest object's id, its type and its score.

In [ ]:
# TODO
body = {
    "size": 1,
    "_source": {"excludes": ["objects.embedding"]},     # boxes and ids, not 1,152 numbers per object
    "query": {"nested": {
        "path": "objects",
        "score_mode": "max",
        "inner_hits": {"size": 1, "_source": False},    # which object in the frame matched
        "query": {"knn": {
            "field": FIXME,                             # the nested field that holds each object's embedding
            "query_vector": FIXME,                      # the embedding of the car above
            "k": 10,
            "num_candidates": 100,
            "filter": {"bool": {"filter": [cs.source_filter(STEM)],                         # this clip only
                                "must_not": [{"term": {"timestamp": QUERY_FRAME_TIME}}]}},  # any frame but the car's own
        }},
    }},
}
hit = cs.es("POST", "/mdx-raw-2025-01-01/_search", json=body)["hits"]["hits"][0]
best = hit["inner_hits"]["objects"]["hits"]["hits"][0]
match = hit["_source"]["objects"][best["_nested"]["offset"]]   # the object of that frame that matched
ANSWER_OBJECT_ID = str(match["id"])
print(f"closest object: {ANSWER_OBJECT_ID}, a {match.get('type')}, in the frame at {hit['_source']['timestamp']}"
      f" (score {best['_score']:.3f})")

In [ ]:
# Check T6: your answer against the same search, run again from the car you were given.
quiz.save("t6", {"query_object_id": QUERY_OBJECT_ID, "answer_object_id": ANSWER_OBJECT_ID,
                 "query_frame_time": QUERY_FRAME_TIME})
passed = quiz.task("p1-t6", "Find the closest object by appearance", quiz.check_t6)

If the answer has the same object id as the car you searched for, the search found the same car in another
frame: the tracker keeps an object's id from frame to frame. Search by appearance finds objects. RT-Embed
(notebook 4) finds events, such as a collision, in short chunks of video.

## T7. Work out what one camera costs RT-VLM in a day

A live camera produces a new 10-second chunk every 10 seconds. Captioning every chunk for 24 hours takes one
request per chunk. One request takes about the `vlm_latency_ms` you measured in T4.

**Your turn:** replace the three `FIXME`s. Use the mean `vlm_latency_ms` of your `chunk_duration=10` captions
from T4. Then compute the requests one camera needs in a day, and the GPU-seconds those requests take if
RT-VLM answers them one at a time. The cell prints the requests and the GPU time in hours.

In [ ]:
# TODO
saved = json.loads((WORK / "t4_captions.json").read_text())     # the captions T4 saved
latencies = [c["vlm_latency_ms"] for c in saved["chunk_duration_10"]]
print("vlm_latency_ms of the 10-second chunks:", latencies)

LATENCY_MS = FIXME            # the mean latency of one 10-second chunk, in milliseconds
REQUESTS_PER_DAY = FIXME      # 10-second chunks from one camera in 24 hours
GPU_SECONDS_PER_DAY = FIXME   # model time for those requests, one at a time, in seconds
print(f"{LATENCY_MS:.0f} ms per chunk: {REQUESTS_PER_DAY} requests and "
      f"{GPU_SECONDS_PER_DAY / 3600:.1f} GPU-hours a day for one camera")

In [ ]:
# Check T7: your latency, requests and GPU-seconds against the latencies T4 recorded.
quiz.save("t7", {"latency_ms": float(LATENCY_MS), "requests_per_day": int(REQUESTS_PER_DAY),
                 "busy_seconds_per_day": float(GPU_SECONDS_PER_DAY)})
passed = quiz.task("p1-t7", "Work out what one camera costs RT-VLM in a day", quiz.check_t7)

This is a one-at-a-time estimate. RT-VLM batches requests that arrive together, so one GPU serves more
cameras than these numbers suggest, and each answer can take longer. Before you size a deployment, measure
latency and throughput with as many cameras as you plan to run. Captioning only the windows that RT-CV or
RT-Embed picked out keeps the request count down.

## Answer the questions

Answer each question in the code cell under it: put one letter between the quotes, then run the cell. A
correct answer prints a short explanation. You can change an answer and run the cell again. The grader reads
your last answer.

**Q1.** A teammate stores `junction.mp4` in VIOS with a single upload request to VIOS itself. They do not use
the VSS agent service. VIOS lists the video and plays it back, but ten minutes later Elasticsearch still holds
no RT-CV records for it. What is the most likely reason?

- **A.** RT-CV skips videos shorter than one minute.
- **B.** Nothing told the VSS agent service that the upload was complete, so RT-CV and RT-Embed never started
  on the file.
- **C.** RT-CV wrote its records under the VIOS sensor id, and the query used the file stem.
- **D.** Kafka drops the messages of videos uploaded in a single chunk.

In [ ]:
correct = quiz.answer("p1-q1", "")     # your letter: A, B, C or D

**Q2.** You upload `junction.mp4` with the three calls. VIOS lists it as `junction_1`, with a UUID as its
sensor id. Which filter selects its RT-CV frame records in `mdx-raw-2025-01-01`?

- **A.** `{"term": {"sensorId.keyword": "junction_1"}}`
- **B.** `{"term": {"sensorId.keyword": "<the UUID>"}}`
- **C.** `{"term": {"sensor.id.keyword": "<the UUID>"}}`
- **D.** `{"term": {"sensorId.keyword": "junction"}}`

In [ ]:
correct = quiz.answer("p1-q2", "")     # your letter: A, B, C or D

**Q3.** You build two features on the same deployment: an alarm that must go off within a second of a new
detection, and a weekly report of last week's detections. Which interface fits each?

- **A.** Kafka for the alarm, Elasticsearch for the weekly report.
- **B.** Elasticsearch for the alarm, Kafka for the weekly report.
- **C.** Kafka for both, because a topic keeps every message forever.
- **D.** Elasticsearch for both, because Kafka carries only video.

In [ ]:
correct = quiz.answer("p1-q3", "")     # your letter: A, B, C or D

**Q4.** Your T7 numbers assume that RT-VLM answers one request at a time. Why can one RT-VLM serve more
cameras than that, and what do you measure before you size a deployment?

- **A.** Short prompts cost almost nothing, so you only need to count the requests.
- **B.** Latency falls as you add cameras, so you measure one camera and multiply.
- **C.** RT-VLM batches requests that arrive together. Total throughput grows, and each answer takes longer. You
  measure latency and throughput with as many cameras as you plan to run.
- **D.** RT-VLM skips chunks in which RT-CV detects nothing, so you only measure RT-CV.

In [ ]:
correct = quiz.answer("p1-q4", "")     # your letter: A, B, C or D

**Q5.** RT-VLM captions a 30-second chunk from 8 frames spread evenly across it. Two cars touch for about one
second. What is the risk, and what helps?

- **A.** There is no risk: the model reads every frame of the chunk.
- **B.** The sampled frames are about 3.75 seconds apart, so the contact can fall between two of them. Sample
  more frames per chunk, or use shorter chunks.
- **C.** RT-VLM ignores the contact unless TrafficCamNet RT-DETR labels it as a collision first.
- **D.** A higher `temperature` makes the model look at more frames.

In [ ]:
correct = quiz.answer("p1-q5", "")     # your letter: A, B, C or D

**Q6.** An alert system checks every 10-second chunk of a live camera. A program reads RT-VLM's answer and
needs a fast yes or no. Which request fits best?

- **A.** "Describe this traffic scene in detail." The program then searches the caption for the word
  "collision".
- **B.** "What's in this video?" with `chunk_duration` set to 0.
- **C.** A yes-or-no question with the reasoning template added, so the model thinks before every answer.
- **D.** "You are a traffic monitoring system. Did two vehicles collide in this clip? Answer with only 'yes'
  or 'no'."

In [ ]:
correct = quiz.answer("p1-q6", "")     # your letter: A, B, C or D

**Q7.** Two requests arrive: "show me white cars" and "show me a collision". Which part of this deployment
answers each one best?

- **A.** White cars: RT-CV, by comparing a text embedding from RT-CV's text encoder with the embeddings of
  the detected objects. A collision: RT-Embed, by comparing a Cosmos-Embed1 text embedding with the embeddings of
  short video chunks.
- **B.** White cars: filter `mdx-raw` on the detector class `white car`. A collision: filter on the detector
  class `collision`.
- **C.** White cars: RT-Embed, because Cosmos-Embed1 sees color. A collision: RT-CV, because TrafficCamNet
  RT-DETR labels collisions.
- **D.** Both: caption every chunk of every video with RT-VLM and search the caption text.

In [ ]:
correct = quiz.answer("p1-q7", "")     # your letter: A, B, C or D

**Q8.** You search the four traffic videos for "a polar bear in a snowstorm" and get five windows, each with a
score. What do these results tell you?

- **A.** At least one of the videos shows a polar bear.
- **B.** The index is broken: an unrelated query should return no results.
- **C.** Nothing on their own. A k-nearest-neighbor search always returns the k closest windows. Compare the
  scores with a real query and watch a clip before you trust a hit.
- **D.** The scores are probabilities: each one is the chance that the window shows a polar bear.

In [ ]:
correct = quiz.answer("p1-q8", "")     # your letter: A, B, C or D

## Check your results

The next cell lists every task check and every answer you have run. T1, T2, T4 and T5 must pass and at least
6 of the 8 answers must be correct, so fix any NOT YET among them before you submit.

In [ ]:
counts = quiz.summary()

## Submit

The Submit cell asks the grader, a separate service, to grade your work. The grader reads `answers.json`,
checks your tasks again against the live services, and compares your answers with its own key. It prints one
line per check. You can submit as often as you like.

When the result says **PASSED**, click **ASSESS TASK** on the course page.

In [ ]:
# Submit: the grader checks answers.json and the live services, then reports one line per check.
import requests
import course_ui

GRADE_URL = "http://assessment:8081/grade/part1"
response = requests.post(GRADE_URL, timeout=300)
if response.status_code != 200:
    raise RuntimeError(f"POST {GRADE_URL} -> HTTP {response.status_code}: {response.text[:500]}")
result = response.json()
print(f"Part 1: {'PASSED' if result['passed'] else 'NOT PASSED'}   graded at {result.get('graded_at')}")
course_ui.results(result, "Part 1")

## Next

Open VSS Coding Agent Skills. It is a page in Course Docs, not a notebook: open the JupyterLab Launcher
(**File > New Launcher**), click **Course Docs**, then choose **VSS Coding Agent Skills**. The page opens in a
JupyterLab tab, and you work in a terminal next to it. There you describe what you did by hand in Part 1 to a
coding agent, and it does the work with the VSS skills.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the Submit cell shows PASSED and you have clicked ASSESS TASK on the course page.
>
> **Next: VSS Coding Agent Skills (open Course Docs from the JupyterLab Launcher)** &middot; Part 1 &middot; about 40 min  
> Have a coding agent with the VSS skills process four traffic videos, report possible collisions with clip links, and watch a live stream for a collision.
>
> [Back to Home](../00_Home.ipynb)


<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>